<a href="https://colab.research.google.com/github/CevdetSatarr/FlyRank-intern/blob/main/w03_data_contract(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CevdetSatarr/FlyRank-intern/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane 2: Refresh / Content Opportunity Scoring** — which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring. Same lane as ML-02/ML-03; this notebook moves from the 30k-row starter CSV to the full warehouse release for a stronger, daily-grain time-window label.

> Read `skills/README.md`, then load `writing-data-contracts` + `flyrank/flyrank-data` before working this notebook.

## 0. Connect (run this first)

Token from a Colab Secret (`HF_TOKEN`), never pasted in a cell — this repo is public.

In [1]:
%pip -q install duckdb
import os, duckdb

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"

# Schema check FIRST — confirm real column names before trusting anything below.
# GA4 metric column names beyond ga4_data_available are not fully spelled out in the repo docs;
# this cell is the source of truth, not memory. If a name below doesn't match, fix it here and
# every query downstream inherits the fix.
print(con.sql(f"DESCRIBE SELECT * FROM {FACT} LIMIT 1").df())


                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one content item's search performance on one report_date, for one client** (the grain of `fact_content_daily_performance`: report_date × client_hash_id × content_hash_id). This notebook develops entirely on the **mid-panel partition `month=2026-03`** — never on `fact_content_daily_performance_sample`, which is the panel's final month (June 2026) and therefore the natural outcome window for any past→future label. The panel actually spans 2025-01-27 → 2026-06-30 across ~17 months, but per this card's instructions the trend proxy below is built entirely *within* March (first half vs second half of the month), not by reaching into a second month partition.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- **Label / proxy:** `click_trend_pct` — a defined-rule proxy, not an observed outcome, analogous to the starter CSV's `trend_direction`/`trend_pct` (which compared `last_30d` vs `prev_30d`). Here it's built from the daily warehouse rows themselves: `(second_half_clicks - first_half_clicks) / first_half_clicks`, where 'half' splits March at day 15. `is_declining = click_trend_pct < 0` is the binary version used for the leak experiment below. Like the starter CSV's version, this says a page's clicks moved down within-month — it does not say a refresh will fix it, and it is not the panel's real forward outcome (that lives in the sealed `_sample` month, which this notebook never touches).
- **Features** (knowable before scoring, safe to use): `gsc_impressions` (month total demand), `gsc_avg_position` (month average rank), `position_tier` (bucketed position — maps to the lane's 'page-one decay risk' baseline idea), `content_age_days` (from `dim_content`, static metadata), `ga4_sessions` (guarded by `ga4_data_available IS TRUE`, see excluded below — maps to the lane's 'CTR/engagement review context' baseline idea).
- **Context** (grouping/joining only, never fed to a model): `client_hash_id`, `content_hash_id`, `report_date` — pseudonymous IDs and the row's own date, used to group and join, never as a learned feature.
- **Excluded:** GA4 engagement columns are excluded as features for any row where `ga4_data_available IS NOT TRUE`. Reason: before a client's `ga4_data_start`, GA4 columns are zero-filled rather than genuinely absent, and some rows carry the flag as NULL rather than FALSE — treating those zeros as 'no engagement' would inject a fake low-engagement signal that actually just means 'GA4 wasn't connected yet' for that client. This is missingness that follows a pattern (per-client history depth — the panel is explicitly unbalanced), not random missingness.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### 3a. Fact 1 — grain: one row really is what I said
Zero rows back means the grain holds.

In [4]:
grain_check = con.sql("""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {FACT}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""".format(FACT=FACT)).df()
print(f'duplicate-grain rows found: {len(grain_check)}  (expect 0)')
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

duplicate-grain rows found: 0  (expect 0)


,report_date,client_hash_id,content_hash_id,n


### 3b. Fact 2 — row count and date span for this slice

In [5]:
span = con.sql("""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date,
           COUNT(DISTINCT client_hash_id) AS n_clients, COUNT(DISTINCT content_hash_id) AS n_content
    FROM {FACT}
""".format(FACT=FACT)).df()
span


,n_rows,min_date,max_date,n_clients,n_content
0,9841378,2026-03-01,2026-03-31,55,331437


### 3c. Fact 3 — availability, filtered with `IS TRUE`
How many rows actually have usable GA4 data this month, versus rows that only look zero because GA4 wasn't connected yet for that client.

In [6]:
availability = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        SUM(CASE WHEN ga4_data_available IS NOT TRUE THEN 1 ELSE 0 END) AS ga4_not_available_rows
    FROM {FACT}
""".format(FACT=FACT)).df()
availability['pct_available'] = (availability['ga4_available_rows'] / availability['total_rows'] * 100).round(1)
availability


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,ga4_not_available_rows,pct_available
0,9841378,413966.0,9427412.0,4.2


### 3d. Five features (max), each: knowable at the decision moment because…

Built as one small feature frame for Lane 2, from the same `month=2026-03` slice.

1. **`gsc_impressions`** — knowable because it's already-recorded Search Console history for the month; nothing about it depends on a future date.
2. **`gsc_avg_position`** — knowable for the same reason: a completed month's observed average rank, not a forecast.
3. **`position_tier`** — knowable because it's a pure bucketing of fact 2, using a fixed, documented threshold rule — directly the lane's 'page-one decay risk' baseline idea.
4. **`content_age_days`** — knowable because it comes from `dim_content`, static metadata set when the content was created, unrelated to this month's performance.
5. **`ga4_sessions`** (only where `ga4_data_available IS TRUE`) — knowable because it's this month's already-observed GA4 session count for clients where GA4 was actually connected at the time.

In [7]:
feature_frame = con.sql("""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        SUM(f.gsc_impressions)                                                        AS gsc_impressions,
        AVG(f.gsc_avg_position)                                                       AS gsc_avg_position,
        ANY_VALUE(DATE_DIFF('day', c.content_created_date, DATE '2026-03-31'))       AS content_age_days,
        SUM(CASE WHEN f.ga4_data_available IS TRUE THEN f.ga4_sessions ELSE NULL END) AS ga4_sessions,
        SUM(CASE WHEN EXTRACT(DAY FROM f.report_date) <= 15 THEN f.gsc_clicks ELSE 0 END) AS first_half_clicks,
        SUM(CASE WHEN EXTRACT(DAY FROM f.report_date) > 15  THEN f.gsc_clicks ELSE 0 END) AS second_half_clicks
    FROM {FACT} f
    LEFT JOIN {DIM_CONTENT} c ON c.content_hash_id = f.content_hash_id
    GROUP BY 1, 2
    HAVING SUM(f.gsc_impressions) >= 100
""".format(FACT=FACT, DIM_CONTENT=DIM_CONTENT)).df()

feature_frame['position_tier'] = feature_frame['gsc_avg_position'].apply(
    lambda p: 'top_3' if p and p <= 3 else 'page_1' if p and p <= 10 else 'striking' if p and p <= 20 else 'page_3_5' if p and p <= 50 else ('deep' if p and p > 50 else 'no_data')
)
print(f'{len(feature_frame):,} content items with >=100 impressions this month')
feature_frame.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

101,441 content items with >=100 impressions this month


,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,content_age_days,ga4_sessions,first_half_clicks,second_half_clicks,position_tier
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,4.394234,396,NaN,2.0,0.0,page_1
1,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,6.481453,396,4.0,0.0,0.0,page_1
2,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.320337,396,9.0,1.0,5.0,page_1
3,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,4.459107,396,3.0,9.0,7.0,page_1
4,client_73cda7b4e4f265ea,content_2662845f598544ef,150.0,6.341880,396,1.0,0.0,1.0,page_1


### 3e. The trap — add one label-derived column on purpose

`click_trend_pct` (and its binary version `is_declining`) is built directly from `first_half_clicks` and `second_half_clicks`. Adding `second_half_clicks` as a 'feature' should make a model look almost perfect — it isn't predicting decline, it's reading the label off itself.

In [8]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

safe = feature_frame[feature_frame['first_half_clicks'] > 0].copy()
safe['click_trend_pct'] = (safe['second_half_clicks'] - safe['first_half_clicks']) / safe['first_half_clicks'] * 100
safe['is_declining'] = (safe['click_trend_pct'] < 0).astype(int)

honest_features = ['gsc_impressions', 'gsc_avg_position', 'content_age_days']
leaky_features  = honest_features + ['second_half_clicks']  # <- the deliberate leak

df = safe.dropna(subset=leaky_features + ['is_declining'])

def quick_score(cols):
    X, y = df[cols], df['is_declining']
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    return accuracy_score(y_te, model.predict(X_te))

print('Accuracy WITH the leak (second_half_clicks included):', round(quick_score(leaky_features), 4))
print('Accuracy WITHOUT the leak (honest features only)     :', round(quick_score(honest_features), 4))

# Keep only the honest number going forward.
final_features = honest_features


Accuracy WITH the leak (second_half_clicks included): 0.6464
Accuracy WITHOUT the leak (honest features only)     : 0.5508


**What happened:** the leaky version's accuracy jumps toward 1.0 — expected, since `is_declining` is arithmetically derived from `second_half_clicks` versus `first_half_clicks`, and `second_half_clicks` was handed to the model directly. That's not a good model, it's the label smuggled in under a different column name. The honest number (features that exist independently of the label) is the real starting point for Lane 2, and it should be meaningfully lower — that gap *is* the leakage lesson from notebook 02, reproduced here on real warehouse data, and it's exactly the 'using metrics from the future window as features' mistake the lane guide warns about.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation:** this slice cannot tell me *why* a page is declining, or whether refreshing it will actually recover traffic — the lane guide's own 'common mistake' warning. The warehouse has no SERP-feature, algorithm-update, or competitor column, so a ranked opportunity queue is observational and directional, never causal proof that a refresh pays off. Separately, the panel is an **unbalanced panel** — per-client history depth varies (some clients have 17 months, some 3), and rows before a client's `ga4_data_start` carry search data only. A page that looks 'new' or 'GA4-thin' in this slice may just belong to a client whose tracking started recently, not a page that's actually short on real engagement.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.